# Dành cho hệ thống tự động điều khiển

In [17]:
# @title 1. Fast Setup (~45s): Pre-built llama-server CUDA + Fast Download Models (LLM 26B + STT E2B)

import os
import shutil
import subprocess
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor

t_start = time.time()
print("⚡ [1/3] Đang cài đặt các thư viện Python & hf_transfer (tăng tốc tải Hugging Face)...")
!pip install pyngrok huggingface_hub hf_transfer fastapi uvicorn httpx python-multipart soundfile scipy numpy --quiet

# Bật hf_transfer (viết bằng Rust) giúp tải GGUF trên Colab đạt 500-800 MB/s
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# --- CẤU HÌNH MODEL LLM CHÍNH (PORT 8001) ---
LLM_HF_REPO = "AlexHung29629/gemma-4-26B-A4B-it-GGUF"
LLM_MODEL_FILE = "gemma-4-26B-A4B-it-Q4_K_M.gguf"

# --- CẤU HÌNH MODEL STT GEMMA 4 E2B (PORT 8003) ---
# Bản GGUF chính thức từ ggml-org có kèm mmproj chứa Audio Conformer (gemma4a) + Vision
STT_HF_REPO = "ggml-org/gemma-4-E2B-it-GGUF"
STT_MODEL_FILE = "gemma-4-E2B-it-Q4_0.gguf"
STT_MMPROJ_FILE = "mmproj-gemma-4-E2B-it-Q8_0.gguf"

# Tùy chọn: Đặt True nếu muốn lưu cache binary vào Google Drive cá nhân
USE_GDRIVE_CACHE = False
# --------------------------------------------------

BIN_DIR = "/content/build/bin"
SERVER_BIN = f"{BIN_DIR}/llama-server"
os.makedirs(BIN_DIR, exist_ok=True)

def verify_cuda_binary() -> bool:
    """Kiểm tra llama-server đã chạy được và nhận diện được GPU CUDA chưa."""
    if not os.path.exists(SERVER_BIN):
        return False
    os.chmod(SERVER_BIN, 0o755)
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = f"{BIN_DIR}:/usr/local/cuda/lib64:" + env.get("LD_LIBRARY_PATH", "")
    try:
        res = subprocess.run(
            [SERVER_BIN, "--list-devices"],
            capture_output=True,
            text=True,
            env=env,
            timeout=15,
        )
        out = (res.stdout or "") + (res.stderr or "")
        if res.returncode == 0 and "CUDA" in out:
            print(f"✅ Kiểm tra llama-server CUDA thành công:\n{out.strip()}")
            return True
    except Exception as e:
        print(f"⚠️ Kiểm tra binary thất bại: {e}")
    return False

def download_and_extract_tar(url: str, label: str) -> bool:
    tar_path = f"/tmp/{label}.tar.gz"
    extract_dir = f"/tmp/extract_{label}"
    shutil.rmtree(extract_dir, ignore_errors=True)
    os.makedirs(extract_dir, exist_ok=True)
    print(f"⬇️ Đang tải {label} từ GitHub Release...")
    ret = subprocess.run(["curl", "-L", "--fail", "-sS", url, "-o", tar_path])
    if ret.returncode != 0:
        return False
    subprocess.run(["tar", "-xzf", tar_path, "-C", extract_dir], check=True)
    # Tìm thư mục chứa các file binary/.so và dùng cp -a để giữ nguyên toàn bộ symlink (.so)
    entries = os.listdir(extract_dir)
    src_dir = os.path.join(extract_dir, entries[0]) if len(entries) == 1 and os.path.isdir(os.path.join(extract_dir, entries[0])) else extract_dir
    subprocess.run(f"cp -a {src_dir}/. {BIN_DIR}/", shell=True, check=True)
    os.remove(tar_path)
    shutil.rmtree(extract_dir, ignore_errors=True)
    return True

print("\n⚡ [2/3] Đang thiết lập llama-server GPU (ưu tiên bản Pre-built CUDA ~10 giây thay vì compile 10 phút)...")
if USE_GDRIVE_CACHE and not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

gdrive_cache_tar = "/content/drive/MyDrive/llama_cache/llama_server_cuda_bin.tar.gz"
if USE_GDRIVE_CACHE and os.path.exists(gdrive_cache_tar) and not verify_cuda_binary():
    print("📦 Đang khôi phục llama-server từ Google Drive cache...")
    subprocess.run(["tar", "-xzf", gdrive_cache_tar, "-C", BIN_DIR], check=False)

if not verify_cuda_binary():
    # Lấy tag nightly mới nhất từ ggml-org/llama.cpp (fallback về b11429 đã kiểm chứng hỗ trợ Gemma 4 Audio)
    release_tag = "b11429"
    try:
        with urllib.request.urlopen(
            "https://github.com/ggml-org/llama.cpp/releases/latest/download/nightly-tag.txt",
            timeout=5,
        ) as resp:
            tag_text = resp.read().decode("utf-8").strip()
            if tag_text.startswith("b"):
                release_tag = tag_text
    except Exception:
        pass

    base_dl = f"https://github.com/ggml-org/llama.cpp/releases/download/{release_tag}"
    # Thử bản Ubuntu CUDA 13.4 trước (nhẹ ~150MB, khớp với CUDA 13 trên Colab mới)
    if download_and_extract_tar(f"{base_dl}/llama-{release_tag}-bin-ubuntu-cuda-13.4-x64.tar.gz", "llama_cuda13"):
        if not verify_cuda_binary():
            download_and_extract_tar(f"{base_dl}/cudart-llama-{release_tag}-bin-ubuntu-cuda-13.4-x64.tar.gz", "cudart13")

    # Nếu driver Colab thấp hơn CUDA 13.4 -> dùng bản Ubuntu CUDA 12.8 + cudart 12.8 (chạy được trên mọi driver CUDA 12 & 13)
    if not verify_cuda_binary():
        print("🔄 Chuyển sang bản Pre-built Ubuntu CUDA 12.8 (tương thích mọi driver Colab)...")
        download_and_extract_tar(f"{base_dl}/llama-{release_tag}-bin-ubuntu-cuda-12.8-x64.tar.gz", "llama_cuda12")
        if not verify_cuda_binary():
            download_and_extract_tar(f"{base_dl}/cudart-llama-{release_tag}-bin-ubuntu-cuda-12.8-x64.tar.gz", "cudart12")

# Fallback cuối cùng: nếu GitHub Release bị chặn mạng thì mới compile bằng Ninja
if not verify_cuda_binary():
    print("⚠️ Không tải được bản Pre-built, chuyển sang compile nhanh với Ninja...")
    !apt-get update -qq && apt-get install -y -qq ninja-build
    !rm -rf llama.cpp build
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp
    !cmake -B build -S llama.cpp -G Ninja -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF
    !cmake --build build --config Release --target llama-server -j$(nproc)
    verify_cuda_binary()

if USE_GDRIVE_CACHE and os.path.exists("/content/drive/MyDrive") and not os.path.exists(gdrive_cache_tar):
    os.makedirs(os.path.dirname(gdrive_cache_tar), exist_ok=True)
    subprocess.run(["tar", "-czf", gdrive_cache_tar, "-C", BIN_DIR, "."], check=False)
    print(f"💾 Đã lưu cache llama-server vào Google Drive: {gdrive_cache_tar}")

print("\n⚡ [3/3] Đang tải song song 2 model GGUF với hf_transfer (500-800 MB/s)...")
from huggingface_hub import hf_hub_download

downloads = [
    (LLM_HF_REPO, LLM_MODEL_FILE),
    (STT_HF_REPO, STT_MODEL_FILE),
    (STT_HF_REPO, STT_MMPROJ_FILE),
]

def fetch_file(item: tuple[str, str]) -> str:
    repo, fname = item
    target = f"/content/{fname}"
    if os.path.exists(target) and os.path.getsize(target) > 1024 * 1024:
        print(f"   ✔ Đã có sẵn {fname}, bỏ qua tải lại.")
        return target
    t0 = time.time()
    print(f"   ⬇️ Bắt đầu tải {fname} từ {repo}...")
    path = hf_hub_download(repo_id=repo, filename=fname, local_dir="/content")
    size_gb = os.path.getsize(path) / (1024 ** 3)
    print(f"   ✅ Hoàn tất {fname} ({size_gb:.2f} GB) trong {time.time() - t0:.1f}s!")
    return path

with ThreadPoolExecutor(max_workers=3) as pool:
    list(pool.map(fetch_file, downloads))

print(f"\n🎉 HOÀN TẤT TOÀN BỘ SETUP TRONG {time.time() - t_start:.1f} GIÂY!")


⚡ [1/3] Đang cài đặt các thư viện Python & hf_transfer (tăng tốc tải Hugging Face)...

⚡ [2/3] Đang thiết lập llama-server GPU (ưu tiên bản Pre-built CUDA ~10 giây thay vì compile 10 phút)...
✅ Kiểm tra llama-server CUDA thành công:
Available devices:
  CUDA0: NVIDIA L4 (22563 MiB, 22369 MiB free)
✅ Kiểm tra llama-server CUDA thành công:
Available devices:
  CUDA0: NVIDIA L4 (22563 MiB, 22369 MiB free)

⚡ [3/3] Đang tải song song 2 model GGUF với hf_transfer (500-800 MB/s)...
   ✔ Đã có sẵn gemma-4-26B-A4B-it-Q4_K_M.gguf, bỏ qua tải lại.
   ✔ Đã có sẵn gemma-4-E2B-it-Q4_0.gguf, bỏ qua tải lại.
   ✔ Đã có sẵn mmproj-gemma-4-E2B-it-Q8_0.gguf, bỏ qua tải lại.

🎉 HOÀN TẤT TOÀN BỘ SETUP TRONG 4.1 GIÂY!


In [18]:
# @title 2. Start Backend Services (Port 8003: Gemma 4 STT & Port 8001: Gemma 4 26B LLM)

import os
import subprocess
import time
import urllib.request

LLM_PORT = "8001"
STT_PORT = "8003"

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
# Đảm bảo llama-server tìm thấy các thư viện .so (libggml-cuda.so, libmtmd.so, libcudart.so)
env["LD_LIBRARY_PATH"] = "/content/build/bin:/usr/local/cuda/lib64:" + env.get("LD_LIBRARY_PATH", "")

server_path = "/content/build/bin/llama-server"
llm_model_path = f"/content/{LLM_MODEL_FILE}"
stt_model_path = f"/content/{STT_MODEL_FILE}"
stt_mmproj_path = f"/content/{STT_MMPROJ_FILE}"

# Đóng các tiến trình llama-server cũ nếu đang chạy để giải phóng VRAM & Port
!pkill -f llama-server || true
time.sleep(2)

subprocess.run([server_path, "--version"], env=env, check=False)

def wait_for_server(port: str, name: str, proc: subprocess.Popen, log_path: str, timeout_s: int = 180) -> None:
    url = f"http://127.0.0.1:{port}/health"
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            tail = ""
            if os.path.exists(log_path):
                with open(log_path, "r", encoding="utf-8", errors="replace") as lf:
                    tail = "".join(lf.readlines()[-25:])
            raise RuntimeError(
                f"❌ {name} (port {port}) đã dừng đột ngột với mã lỗi {proc.returncode}!\nLog cuối:\n{tail}"
            )
        try:
            with urllib.request.urlopen(url, timeout=2) as resp:
                if resp.status == 200:
                    print(f"✅ {name} đã sẵn sàng tại http://127.0.0.1:{port} ({time.time() - t0:.1f}s)")
                    return
        except Exception:
            pass
        time.sleep(1.5)
    raise TimeoutError(f"❌ Quá thời gian chờ {name} khởi động trên port {port}.")

# -------------------------------------------------------------------------
# BƯỚC 1: Khởi động Gemma 4 E2B STT trên Port 8003 TRƯỚC (~3.3 GB VRAM)
# Khởi động trước để Audio Encoder (mmproj) + E2B luôn nằm 100% trên GPU.
# -------------------------------------------------------------------------
stt_log_path = "/content/stt_8003.log"
stt_log_file = open(stt_log_path, "w", encoding="utf-8")
stt_command = [
    server_path,
    "-m", stt_model_path,
    "--mmproj", stt_mmproj_path,
    "--port", STT_PORT,
    "--alias", "gemma-4-stt,google/gemma-4-E2B-it",
    "--temp", "0.0",
    "--top-p", "1.0",
    "--flash-attn", "on",
    "--kv-unified",
    "--cache-type-k", "q8_0",
    "--cache-type-v", "q8_0",
    "-ngl", "99",
    "--ctx-size", "4096",
    "--parallel", "1",
    "--cont-batching",
    "--jinja",
    "--reasoning", "off",
]

print(f"🎙️ Đang khởi động STT Backend (Gemma 4 E2B + Audio mmproj) trên port {STT_PORT}...")
stt_process = subprocess.Popen(
    stt_command,
    stdout=stt_log_file,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    env=env,
    cwd="/content",
)
wait_for_server(STT_PORT, "STT Backend (Gemma 4 E2B)", stt_process, stt_log_path)

# -------------------------------------------------------------------------
# BƯỚC 2: Khởi động Main LLM (Gemma 4 26B A4B) trên Port 8001
# Dùng --fit on để llama-server tự động cân đối VRAM còn lại trên L4 24GB / A100.
# -------------------------------------------------------------------------
llm_log_path = "/content/llm_8001.log"
llm_log_file = open(llm_log_path, "w", encoding="utf-8")
llm_command = [
    server_path,
    "-m", llm_model_path,
    "--port", LLM_PORT,
    "--alias", "main-llm,gemma-4-26b,gemma-4-26B-A4B-it-Q4_K_M.gguf",
    "--seed", "3407",
    "--temp", "0.6",
    "--top-p", "0.95",
    "--top-k", "20",
    "--min-p", "0.0",
    "--repeat-penalty", "1.0",
    "--flash-attn", "on",
    "--kv-unified",
    "--cache-type-k", "q8_0",
    "--cache-type-v", "q8_0",
    "--batch-size", "2048",
    "--ubatch-size", "512",
    "-ngl", "99",
    "--fit", "on",
    "--fit-target", "512",
    "--ctx-size", "65536",
    "--parallel", "1",
    "--cont-batching",
    "--jinja",
    "--reasoning", "off",
]

print(f"🤖 Đang khởi động Main LLM Backend (Gemma 4 26B) trên port {LLM_PORT}...")
llm_process = subprocess.Popen(
    llm_command,
    stdout=llm_log_file,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    env=env,
    cwd="/content",
)
wait_for_server(LLM_PORT, "Main LLM Backend (Gemma 4 26B)", llm_process, llm_log_path)

print("\n📊 Trạng thái VRAM sau khi nạp cả 2 model:")
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv


^C
🎙️ Đang khởi động STT Backend (Gemma 4 E2B + Audio mmproj) trên port 8003...
✅ STT Backend (Gemma 4 E2B) đã sẵn sàng tại http://127.0.0.1:8003 (4.5s)
🤖 Đang khởi động Main LLM Backend (Gemma 4 26B) trên port 8001...
✅ Main LLM Backend (Gemma 4 26B) đã sẵn sàng tại http://127.0.0.1:8001 (7.5s)

📊 Trạng thái VRAM sau khi nạp cả 2 model:
name, memory.used [MiB], memory.total [MiB], utilization.gpu [%]
NVIDIA L4, 20008 MiB, 23034 MiB, 0 %


In [19]:
# @title 3. Start FastAPI API Gateway on Port 8000 (Secured: /v1/chat/completions + /v1/audio/transcriptions + /v1/models)

GATEWAY_PORT = "8000"

gateway_code = r'''
from __future__ import annotations

import base64
from collections import defaultdict
import datetime
import hmac
import io
import json
import math
import os
import re
import subprocess
import tempfile
import threading
import time
import unicodedata
import wave
from typing import Any

import httpx
import numpy as np
import soundfile as sf
from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, PlainTextResponse, Response, StreamingResponse
from scipy.signal import resample_poly

LLM_BACKEND = os.environ.get("LLM_BACKEND_URL", "http://127.0.0.1:8001")
STT_BACKEND = os.environ.get("STT_BACKEND_URL", "http://127.0.0.1:8003")
LLM_MODEL_NAME = os.environ.get("LLM_MODEL_FILE", "gemma-4-26B-A4B-it-Q4_K_M.gguf")
STT_MODEL_NAME = os.environ.get("STT_MODEL_FILE", "gemma-4-E2B-it-Q4_0.gguf")
GATEWAY_API_KEY = os.environ.get("GATEWAY_API_KEY", "").strip()

MAX_AUDIO_BYTES = 25 * 1024 * 1024  # Giới hạn 25 MB cho file âm thanh upload
MAX_CHAT_BODY_BYTES = 4 * 1024 * 1024  # Giới hạn 4 MB cho JSON body chat

ALLOWED_WEB_ORIGINS = {
    "https://gloryvocab.com",
    "https://www.gloryvocab.com",
    "http://localhost:3000",
    "http://localhost:5173",
    "http://127.0.0.1:3000",
    "http://127.0.0.1:5173",
}

BLOCKED_UA_KEYWORDS = (
    "masscan",
    "zgrab",
    "censys",
    "shodan",
    "nuclei",
    "nmap",
    "nikto",
    "dirbuster",
    "gobuster",
    "sqlmap",
)

# Rate Limiter: max 60 requests/minute per client IP
RATE_LIMIT_MAX_REQUESTS = 60
RATE_LIMIT_WINDOW_SECONDS = 60.0
_rate_limit_lock = threading.Lock()
_ip_request_timestamps: dict[str, list[float]] = defaultdict(list)


def check_rate_limit(client_ip: str) -> None:
    now = time.time()
    cutoff = now - RATE_LIMIT_WINDOW_SECONDS
    with _rate_limit_lock:
        timestamps = _ip_request_timestamps[client_ip]
        valid_timestamps = [t for t in timestamps if t > cutoff]
        if len(valid_timestamps) >= RATE_LIMIT_MAX_REQUESTS:
            _ip_request_timestamps[client_ip] = valid_timestamps
            raise HTTPException(status_code=429, detail="Rate limit exceeded (max 60 req/min)")
        valid_timestamps.append(now)
        _ip_request_timestamps[client_ip] = valid_timestamps


def get_client_ip(request: Request) -> str:
    cf_ip = request.headers.get("cf-connecting-ip")
    if cf_ip:
        return cf_ip.strip()
    x_forwarded = request.headers.get("x-forwarded-for")
    if x_forwarded:
        return x_forwarded.split(",")[0].strip()
    if request.client and request.client.host:
        return request.client.host
    return "127.0.0.1"


STT_ALIASES = {
    "gemma-4-stt",
    "google/gemma-4-e2b-it",
    "gemma-4-e2b-it",
    STT_MODEL_NAME.lower(),
    "whisper-1",
}

DEFAULT_VI_STT_INSTRUCTION = (
    "Hãy phiên âm chính xác đoạn âm thanh tiếng Việt này sang văn bản tiếng Việt. "
    "Chỉ trả về đúng câu được nói, không dịch sang ngôn ngữ khác, không thêm giải thích hay nhận xét."
)

# Tắt /docs, /redoc, /openapi.json để tránh bot tự động dò quét schema API
app = FastAPI(
    title="GloryVocab Unified Gateway",
    version="1.0.0",
    docs_url=None,
    redoc_url=None,
    openapi_url=None,
)

# Chỉ cho phép CORS từ tên miền chính chủ gloryvocab.com và localhost dev (loại bỏ "*")
app.add_middleware(
    CORSMiddleware,
    allow_origins=list(ALLOWED_WEB_ORIGINS),
    allow_credentials=True,
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["Authorization", "Content-Type", "Accept", "Origin", "X-Requested-With"],
)


@app.middleware("http")
async def security_and_logging_middleware(request: Request, call_next):
    client_ip = get_client_ip(request)
    start_time = time.perf_counter()

    # Rate limiting on all non-health routes
    if request.url.path not in ("/", "/health", "/v1/health"):
        try:
            check_rate_limit(client_ip)
        except HTTPException as exc:
            latency_ms = round((time.perf_counter() - start_time) * 1000.0, 2)
            log_entry = {
                "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                "client_ip": client_ip,
                "method": request.method,
                "path": request.url.path,
                "status_code": exc.status_code,
                "latency_ms": latency_ms,
            }
            print(json.dumps(log_entry), flush=True)
            return JSONResponse(status_code=exc.status_code, content={"detail": exc.detail})

    status_code = 500
    try:
        response = await call_next(request)
        status_code = response.status_code
        return response
    except HTTPException as exc:
        status_code = exc.status_code
        raise
    except Exception:
        status_code = 500
        raise
    finally:
        latency_ms = round((time.perf_counter() - start_time) * 1000.0, 2)
        log_entry = {
            "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "client_ip": client_ip,
            "method": request.method,
            "path": request.url.path,
            "status_code": status_code,
            "latency_ms": latency_ms,
        }
        print(json.dumps(log_entry), flush=True)


def verify_request_access(request: Request) -> None:
    # Chặn các trình duyệt từ trang web lạ, bot quét lỗ hổng và kiểm tra API Key nếu được cấu hình
    ua = (request.headers.get("user-agent") or "").lower()
    if any(b in ua for b in BLOCKED_UA_KEYWORDS):
        raise HTTPException(status_code=403, detail="Forbidden")

    origin = (request.headers.get("origin") or "").rstrip("/").lower()
    if origin and origin not in ALLOWED_WEB_ORIGINS:
        raise HTTPException(status_code=403, detail="Origin not allowed")

    if GATEWAY_API_KEY:
        auth = request.headers.get("authorization") or ""
        token = auth[7:].strip() if auth.lower().startswith("bearer ") else ""
        if not token or not hmac.compare_digest(token, GATEWAY_API_KEY):
            raise HTTPException(status_code=401, detail="Invalid or missing API Key")


def clean_stt_text(raw: str) -> str:
    if not raw:
        return ""
    text = unicodedata.normalize("NFC", raw)
    text = re.sub(r"<\|channel>thought.*?<channel\|>", "", text, flags=re.DOTALL)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|turn>\w*\s*", "", text)
    text = re.sub(r"<turn\|>", "", text)
    text = re.sub(
        r"^(?:Here is the transcription|Transcription|Phiên âm|Nội dung âm thanh|Văn bản)\s*:\s*",
        "",
        text.strip(),
        flags=re.IGNORECASE,
    )
    text = text.strip()
    if len(text) >= 2 and ((text[0] == '"' and text[-1] == '"') or (text[0] == "'" and text[-1] == "'")):
        text = text[1:-1].strip()
    return " ".join(text.split())


def normalize_audio_to_16k_wav(raw_bytes: bytes, filename: str = "audio.wav") -> tuple[bytes, float, float]:
    # Convert uploaded audio bytes into 16kHz mono 16-bit PCM WAV bytes. Returns (wav_bytes, rms_energy, duration_s).
    data: np.ndarray | None = None
    sr: int = 16000
    try:
        data, sr = sf.read(io.BytesIO(raw_bytes), dtype="float32", always_2d=True)
    except Exception:
        # Fallback qua ffmpeg nếu file upload là mp3/m4a/webm/ogg
        suffix = os.path.splitext(filename or "audio.wav")[1] or ".bin"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as src_f:
            src_f.write(raw_bytes)
            src_path = src_f.name
        dst_path = src_path + ".16k.wav"
        try:
            subprocess.run(
                ["ffmpeg", "-y", "-i", src_path, "-ac", "1", "-ar", "16000", "-f", "wav", dst_path],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            data, sr = sf.read(dst_path, dtype="float32", always_2d=True)
        finally:
            for p in (src_path, dst_path):
                try:
                    os.remove(p)
                except OSError:
                    pass

    mono = data.mean(axis=1).astype(np.float32) if data.shape[1] > 1 else data[:, 0].astype(np.float32)
    if sr != 16000 and mono.size > 0:
        g = math.gcd(int(sr), 16000)
        mono = resample_poly(mono, 16000 // g, int(sr) // g).astype(np.float32)

    rms = float(np.sqrt(np.mean(np.square(mono)))) if mono.size > 0 else 0.0
    duration_s = float(mono.size) / 16000.0

    clipped = np.clip(mono, -1.0, 1.0)
    pcm_int16 = (clipped * 32767.0).astype(np.int16)
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(16000)
        wf.writeframes(pcm_int16.tobytes())
    return buf.getvalue(), rms, duration_s


def build_stt_prompt(custom_prompt: str) -> str:
    p = (custom_prompt or "").strip()
    if not p:
        return DEFAULT_VI_STT_INSTRUCTION
    low = p.lower()
    if "phiên âm" in low or "transcribe" in low:
        return p
    # Nếu prompt là danh sách từ vựng gợi ý kiểu Whisper (initial_prompt)
    return f"{DEFAULT_VI_STT_INSTRUCTION} Từ vựng gợi ý (nếu xuất hiện trong âm thanh): {p}"


def is_stt_chat_request(body: dict[str, Any]) -> bool:
    model = str(body.get("model", "")).strip().lower()
    if model in STT_ALIASES or "stt" in model or "e2b" in model:
        return True
    for msg in body.get("messages", []):
        content = msg.get("content") if isinstance(msg, dict) else None
        if isinstance(content, list):
            for part in content:
                if isinstance(part, dict) and part.get("type") in ("input_audio", "audio"):
                    return True
    return False


@app.get("/")
async def root_index() -> dict[str, str]:
    # Không lộ thông tin llm_backend / stt_backend ở trang chủ công khai
    return {"status": "ok"}


@app.get("/health")
@app.get("/v1/health")
async def health(request: Request) -> dict[str, Any]:
    # Nếu gọi từ bên ngoài qua Cloudflare, chỉ trả status tối giản tránh lộ cấu trúc nội bộ
    if request.headers.get("cf-ray"):
        return {"status": "ok"}
    status: dict[str, Any] = {"gateway": "ok", "llm_backend": False, "stt_backend": False}
    async with httpx.AsyncClient(timeout=3.0) as client:
        for key, url in (("llm_backend", LLM_BACKEND), ("stt_backend", STT_BACKEND)):
            try:
                r = await client.get(f"{url}/health")
                status[key] = (r.status_code == 200)
            except Exception:
                status[key] = False
    return status


@app.get("/models")
@app.get("/v1/models")
async def list_models(request: Request) -> dict[str, Any]:
    verify_request_access(request)
    now = int(time.time())
    # Đặt model LLM 26B lên đầu để các client chat tự động (model='auto') chọn LLM chính,
    # và liệt kê rõ gemma-4-stt + google/gemma-4-E2B-it cho STT client.
    models = [
        {
            "id": LLM_MODEL_NAME,
            "object": "model",
            "created": now,
            "owned_by": "google-gemma-4-llm",
        },
        {
            "id": "main-llm",
            "object": "model",
            "created": now,
            "owned_by": "google-gemma-4-llm",
        },
        {
            "id": "gemma-4-stt",
            "object": "model",
            "created": now,
            "owned_by": "google-gemma-4-stt",
        },
        {
            "id": "google/gemma-4-E2B-it",
            "object": "model",
            "created": now,
            "owned_by": "google-gemma-4-stt",
        },
    ]
    return {"object": "list", "data": models}


@app.get("/v1/audio/transcriptions")
async def transcriptions_info(request: Request) -> dict[str, Any]:
    verify_request_access(request)
    return {
        "status": "ready",
        "endpoint": "/v1/audio/transcriptions",
        "model": "gemma-4-stt",
        "base_model": "google/gemma-4-E2B-it",
    }


@app.post("/v1/audio/transcriptions")
async def create_transcription(
    request: Request,
    file: UploadFile = File(...),
    model: str = Form("gemma-4-stt"),
    language: str = Form("vi"),
    prompt: str = Form(""),
    response_format: str = Form("json"),
    temperature: float = Form(0.0),
) -> Response:
    verify_request_access(request)
    t0 = time.perf_counter()
    raw_bytes = await file.read(MAX_AUDIO_BYTES + 1)
    if not raw_bytes:
        raise HTTPException(status_code=400, detail="File âm thanh rỗng.")
    if len(raw_bytes) > MAX_AUDIO_BYTES:
        raise HTTPException(status_code=413, detail="File âm thanh vượt quá giới hạn 25MB.")

    try:
        wav_bytes, rms, duration_s = normalize_audio_to_16k_wav(raw_bytes, file.filename or "voice.wav")
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Không thể đọc/chuẩn hóa file âm thanh: {exc}") from exc

    # Bỏ qua nếu đoạn âm thanh hoàn toàn im lặng để tránh hallucination
    if rms < 1e-4:
        if response_format == "text":
            return PlainTextResponse("")
        return JSONResponse({"text": ""})

    b64_wav = base64.b64encode(wav_bytes).decode("ascii")
    instruction = build_stt_prompt(prompt)

    chat_payload = {
        "model": "gemma-4-stt",
        "temperature": float(temperature),
        "top_p": 1.0,
        "max_tokens": 256,
        "stream": False,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_audio",
                        "input_audio": {
                            "data": b64_wav,
                            "format": "wav",
                        },
                    },
                    {
                        "type": "text",
                        "text": instruction,
                    },
                ],
            }
        ],
    }

    async with httpx.AsyncClient(timeout=60.0) as client:
        resp = await client.post(f"{STT_BACKEND}/v1/chat/completions", json=chat_payload)
        if resp.status_code >= 400:
            raise HTTPException(
                status_code=resp.status_code,
                detail=f"STT backend error: {resp.text}",
            )
        data = resp.json()

    choices = data.get("choices", [])
    raw_text = choices[0].get("message", {}).get("content", "") if choices else ""
    cleaned_text = clean_stt_text(str(raw_text))
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    print(
        f"[STT] file={file.filename} dur={duration_s:.2f}s rms={rms:.4f} "
        f"latency={elapsed_ms:.0f}ms -> {cleaned_text!r}",
        flush=True,
    )

    if response_format == "text":
        return PlainTextResponse(cleaned_text)
    if response_format == "verbose_json":
        return JSONResponse(
            {
                "task": "transcribe",
                "language": language,
                "duration": round(duration_s, 3),
                "text": cleaned_text,
            }
        )
    return JSONResponse({"text": cleaned_text})


@app.post("/v1/chat/completions")
@app.post("/chat/completions")
async def chat_completions(request: Request) -> Response:
    verify_request_access(request)
    raw_body = await request.body()
    if len(raw_body) > MAX_CHAT_BODY_BYTES:
        raise HTTPException(status_code=413, detail="Request body too large.")
    try:
        body = json.loads(raw_body)
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid JSON body: {exc}") from exc

    use_stt = is_stt_chat_request(body)
    target_base = STT_BACKEND if use_stt else LLM_BACKEND
    target_url = f"{target_base}/v1/chat/completions"
    is_stream = bool(body.get("stream", False))

    if is_stream:
        client = httpx.AsyncClient(timeout=httpx.Timeout(900.0, connect=30.0))
        req = client.build_request(
            "POST",
            target_url,
            content=raw_body,
            headers={"Content-Type": "application/json"},
        )
        try:
            resp = await client.send(req, stream=True)
        except httpx.HTTPError as exc:
            await client.aclose()
            return JSONResponse(
                {"error": {"message": f"Backend connection error: {exc}", "type": "server_error"}},
                status_code=502,
            )
        if resp.status_code >= 400:
            err_body = await resp.aread()
            await resp.aclose()
            await client.aclose()
            return Response(
                content=err_body,
                status_code=resp.status_code,
                media_type=resp.headers.get("content-type", "application/json"),
            )

        async def event_stream():
            try:
                async for chunk in resp.aiter_bytes():
                    yield chunk
            finally:
                await resp.aclose()
                await client.aclose()

        return StreamingResponse(
            event_stream(),
            status_code=resp.status_code,
            media_type="text/event-stream",
            headers={
                "Cache-Control": "no-cache",
                "Connection": "keep-alive",
                "X-Accel-Buffering": "no",
            },
        )

    async with httpx.AsyncClient(timeout=300.0) as client:
        try:
            resp = await client.post(
                target_url,
                content=raw_body,
                headers={"Content-Type": "application/json"},
            )
        except httpx.HTTPError as exc:
            return JSONResponse(
                {"error": {"message": f"Backend connection error: {exc}", "type": "server_error"}},
                status_code=502,
            )
        return Response(
            content=resp.content,
            status_code=resp.status_code,
            media_type=resp.headers.get("content-type", "application/json"),
        )
'''

with open("/content/gateway.py", "w", encoding="utf-8") as f:
    f.write(gateway_code.lstrip())

!pkill -f "uvicorn gateway:app" || true
time.sleep(1)

gateway_env = os.environ.copy()
gateway_env["PYTHONUNBUFFERED"] = "1"
gateway_env["LLM_BACKEND_URL"] = f"http://127.0.0.1:{LLM_PORT}"
gateway_env["STT_BACKEND_URL"] = f"http://127.0.0.1:{STT_PORT}"
gateway_env["LLM_MODEL_FILE"] = LLM_MODEL_FILE
gateway_env["STT_MODEL_FILE"] = STT_MODEL_FILE

gateway_log_path = "/content/gateway_8000.log"
gateway_log_file = open(gateway_log_path, "w", encoding="utf-8")
gateway_process = subprocess.Popen(
    ["uvicorn", "gateway:app", "--host", "127.0.0.1", "--port", GATEWAY_PORT],
    stdout=gateway_log_file,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    env=gateway_env,
    cwd="/content",
)

wait_for_server(GATEWAY_PORT, "FastAPI API Gateway", gateway_process, gateway_log_path, timeout_s=30)
print("✅ FastAPI Gateway (Secured) đã sẵn sàng trên Port 8000:")
print("   - GET  /v1/models               -> Liệt kê cả LLM 26B + gemma-4-stt")
print("   - POST /v1/chat/completions     -> Định tuyến tới LLM 26B (8001) hoặc STT E2B (8003)")
print("   - POST /v1/audio/transcriptions -> Nhận file WAV/MP3 tiếng Việt (tối đa 25MB) -> Gemma 4 E2B STT (8003)")


^C
✅ FastAPI API Gateway đã sẵn sàng tại http://127.0.0.1:8000 (1.6s)
✅ FastAPI Gateway (Secured) đã sẵn sàng trên Port 8000:
   - GET  /v1/models               -> Liệt kê cả LLM 26B + gemma-4-stt
   - POST /v1/chat/completions     -> Định tuyến tới LLM 26B (8001) hoặc STT E2B (8003)
   - POST /v1/audio/transcriptions -> Nhận file WAV/MP3 tiếng Việt (tối đa 25MB) -> Gemma 4 E2B STT (8003)


In [ ]:
# @title 4. Publish Public URL (Cloudflare Tunnel api.gloryvocab.com / Gradio.live / Ngrok) & Stream Live Logs

import inspect
import json
import os
import re
import secrets
import subprocess
import sys
import time
from urllib.parse import urlparse

import httpx
from google.colab import userdata
from huggingface_hub import HfApi

# ============================== CẤU HÌNH ==============================
# - "cloudflare": tên miền cố định https://api.gloryvocab.com (khuyên dùng)
# - "gradio": link *.gradio.live ngẫu nhiên
# - "ngrok": Ngrok (cần NGROK_TOKEN trong Secrets)
TUNNEL_PROVIDER = "cloudflare"  # @param ["cloudflare", "gradio", "ngrok"]
CLOUDFLARE_HOSTNAME = "https://api.gloryvocab.com"  # @param {type:"string"}
# KHUYÊN DÙNG: để TRỐNG ô này và lưu token trong 🔑 Secrets (tên CLOUDFLARE_TOKEN, bật "Notebook access").
# Chấp nhận: Tunnel connector token (eyJ...) hoặc Cloudflare API token.
CLOUDFLARE_TOKEN_INPUT = ""  # @param {type:"string"}
# Nếu tên miền cố định không lên được -> tự dùng link tạm *.trycloudflare.com (không cần token, URL đổi mỗi lần chạy)
FALLBACK_TO_QUICK_TUNNEL = True  # @param {type:"boolean"}
# Để TRỐNG = tự dùng repo "<username Hugging Face của bạn>/claw-endpoint" (cần HF_TOKEN quyền Write; không có thì bỏ qua)
DISCOVERY_HF_REPO = ""  # @param {type:"string"}
DISCOVERY_FILE_IN_REPO = "claw_endpoint.json"

CF_API = "https://api.cloudflare.com/client/v4"
CF_BIN = "/content/cloudflared"
CF_LOG = "/content/cloudflared.log"
CF_QUICK_LOG = "/content/cloudflared_quick.log"
CF_TUNNEL_NAME = "models-colab"

# Các biến này do cell 2/3 tạo ra; đặt mặc định để cell 4 vẫn chạy được sau khi khởi động lại runtime
GATEWAY_PORT = globals().get("GATEWAY_PORT", "8000")
LLM_MODEL_FILE = globals().get("LLM_MODEL_FILE", "gemma-4-26B-A4B-it-Q4_K_M.gguf")


# ============================== TIỆN ÍCH ==============================
class CloudflareAPIError(RuntimeError):
    pass


def redact(text) -> str:
    """Che mọi chuỗi giống tunnel token trước khi in ra màn hình."""
    return re.sub(r"eyJ[A-Za-z0-9_\-=]{30,}", "eyJ***", str(text))


def read_secret(name: str) -> str:
    try:
        value = userdata.get(name) or ""
    except Exception:
        value = ""
    return value or os.environ.get(name, "")


def classify_cloudflare_token(raw: str):
    """Trả về ('tunnel'|'api', token) nếu hợp lệ về định dạng, ngược lại (None, lý do)."""
    raw = raw or ""
    m = re.search(r"(eyJ[A-Za-z0-9_\-=]{30,})", raw)
    if m:
        return "tunnel", m.group(1)
    t = raw.strip()
    t = re.sub(r"^(?:authorization\s*:\s*)?bearer\s+", "", t, flags=re.IGNORECASE)
    t = t.strip().strip("\"'`")
    t = re.sub(r"[\s\u200b-\u200d\ufeff]+", "", t)
    if re.fullmatch(r"[A-Za-z0-9_\-]{30,}", t):
        return "api", t
    return None, f"dài {len(t)} ký tự; token thật dài ≥ 30 ký tự, chỉ gồm chữ/số/_/-"


def cloudflare_token_candidates():
    """Gom các token hợp lệ về định dạng: ưu tiên ô nhập tay, sau đó Colab Secret / biến môi trường."""
    sources = [
        ("ô CLOUDFLARE_TOKEN_INPUT", CLOUDFLARE_TOKEN_INPUT),
        ("Colab Secret CLOUDFLARE_TOKEN", read_secret("CLOUDFLARE_TOKEN")),
    ]
    good = []
    for label, raw in sources:
        if not (raw or "").strip():
            continue
        kind, value = classify_cloudflare_token(raw)
        if kind is None:
            print(f"⚠️ Bỏ qua {label}: không phải token Cloudflare hợp lệ ({value}).")
            continue
        if any(v == value for _, _, v in good):
            continue
        good.append((label, kind, value))
    return good


def cf_request(client: httpx.Client, method: str, path: str, **kwargs) -> dict:
    resp = client.request(method, f"{CF_API}{path}", **kwargs)
    try:
        data = resp.json()
    except Exception:
        raise CloudflareAPIError(f"HTTP {resp.status_code}, phản hồi không phải JSON: {resp.text[:200]}") from None
    if not data.get("success"):
        errs = "; ".join(f"[{e.get('code')}] {e.get('message')}" for e in (data.get("errors") or []))
        raise CloudflareAPIError(errs or f"HTTP {resp.status_code}")
    return data


def resolve_cloudflare_connector_token(kind: str, value: str, hostname_url: str, local_port: int) -> str:
    """Tunnel token (eyJ...) dùng trực tiếp. API token -> tự tạo/lấy Tunnel, route và DNS CNAME rồi trả về connector token.

    API token cần quyền: Account > Cloudflare Tunnel: Edit, Zone > DNS: Edit, Zone > Zone: Read (đúng zone gloryvocab.com).
    """
    if kind == "tunnel":
        return value

    parsed = urlparse(hostname_url if "://" in hostname_url else f"https://{hostname_url}")
    full_host = parsed.hostname or "api.gloryvocab.com"
    parts = full_host.split(".")
    headers = {"Authorization": f"Bearer {value}", "Content-Type": "application/json"}
    print(f"🔧 Đang cấu hình Cloudflare Tunnel & DNS cho '{full_host}' qua Cloudflare API...")

    with httpx.Client(headers=headers, timeout=20.0) as client:
        # 1. Tìm zone (thử từ tên đầy đủ rút dần về domain gốc)
        zone_obj = None
        for i in range(max(len(parts) - 1, 1)):
            cand = ".".join(parts[i:])
            found = cf_request(client, "GET", "/zones", params={"name": cand}).get("result") or []
            if found:
                zone_obj = found[0]
                break
        if zone_obj is None:
            raise CloudflareAPIError(
                f"Token hợp lệ nhưng không thấy tên miền của '{full_host}'. Kiểm tra: domain đã nằm trong tài khoản Cloudflare này, "
                "token có quyền Zone:Read + DNS:Edit cho zone đó và Account:Cloudflare Tunnel:Edit."
            )
        zone_id = zone_obj["id"]
        account_id = zone_obj["account"]["id"]

        # 2. Tìm hoặc tạo Tunnel
        listed = cf_request(
            client, "GET", f"/accounts/{account_id}/cfd_tunnel",
            params={"name": CF_TUNNEL_NAME, "is_deleted": "false"},
        ).get("result") or []
        if listed:
            tunnel_id = listed[0]["id"]
        else:
            created = cf_request(
                client, "POST", f"/accounts/{account_id}/cfd_tunnel",
                json={"name": CF_TUNNEL_NAME, "config_src": "cloudflare"},
            )
            tunnel_id = created["result"]["id"]

        # 3. Route: hostname -> http://127.0.0.1:<port>
        cf_request(
            client, "PUT", f"/accounts/{account_id}/cfd_tunnel/{tunnel_id}/configurations",
            json={"config": {"ingress": [
                {"hostname": full_host, "service": f"http://127.0.0.1:{local_port}"},
                {"service": "http_status:404"},
            ]}},
        )

        # 4. DNS CNAME -> <tunnel_id>.cfargotunnel.com
        dns_body = {
            "type": "CNAME",
            "name": full_host,
            "content": f"{tunnel_id}.cfargotunnel.com",
            "proxied": True,
            "ttl": 1,
        }
        records = cf_request(client, "GET", f"/zones/{zone_id}/dns_records", params={"name": full_host}).get("result") or []
        records = [r for r in records if r.get("type") in ("A", "AAAA", "CNAME")]
        if records:
            cf_request(client, "PUT", f"/zones/{zone_id}/dns_records/{records[0]['id']}", json=dns_body)
        else:
            cf_request(client, "POST", f"/zones/{zone_id}/dns_records", json=dns_body)

        # 5. Connector token để chạy cloudflared
        tok = cf_request(client, "GET", f"/accounts/{account_id}/cfd_tunnel/{tunnel_id}/token").get("result")
        if not tok:
            raise CloudflareAPIError("Không lấy được Tunnel connector token.")
        print(f"✅ Đã cấu hình xong Tunnel '{CF_TUNNEL_NAME}' + DNS CNAME '{full_host}'.")
        return str(tok).strip()


def ensure_cloudflared() -> None:
    if os.path.exists(CF_BIN) and os.path.getsize(CF_BIN) > 1_000_000:
        return
    print("⬇️ Đang tải cloudflared binary cho Linux x64...")
    subprocess.run(
        ["curl", "-L", "--fail", "-sS",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-o", CF_BIN],
        check=True,
    )
    os.chmod(CF_BIN, 0o755)


def read_cf_log(log_path: str = CF_LOG) -> str:
    try:
        with open(log_path, "r", encoding="utf-8", errors="replace") as lf:
            return lf.read()
    except OSError:
        return ""


def build_selftest_headers() -> dict:
    headers = {"User-Agent": "colab-selftest/1.0"}
    api_key = (read_secret("GATEWAY_API_KEY") or "").strip()
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    cf_id = (read_secret("CF_ACCESS_CLIENT_ID") or "").strip()
    cf_secret = (read_secret("CF_ACCESS_CLIENT_SECRET") or "").strip()
    if cf_id and cf_secret:
        headers["CF-Access-Client-Id"] = cf_id
        headers["CF-Access-Client-Secret"] = cf_secret
    return headers


def probe_public_models_api(public_url: str, timeout_s: float = 8.0) -> tuple[str, str]:
    """Kiểm tra /v1/models KHÔNG tự động follow redirect sang trang login HTML."""
    url = f"{public_url.rstrip('/')}/v1/models"
    r = httpx.get(url, headers=build_selftest_headers(), timeout=timeout_s, follow_redirects=False)
    www_auth = (r.headers.get("www-authenticate") or "").lower()
    location = r.headers.get("location") or ""
    loc_low = location.lower()
    if "cloudflare-access" in www_auth or "cloudflareaccess.com" in loc_low or "/cdn-cgi/access/login" in loc_low:
        return "cloudflare_access", (location or "Cloudflare Zero Trust Access")
    if r.status_code in (301, 302, 303, 307, 308):
        return "redirect", f"HTTP {r.status_code} -> {location}"
    ctype = (r.headers.get("content-type") or "").lower()
    body_head = r.text[:200].lstrip().lower()
    if "text/html" in ctype or body_head.startswith(("<!doctype", "<html")):
        return "html", f"HTTP {r.status_code} trả về trang HTML thay vì JSON"
    if r.status_code == 200:
        try:
            data = r.json()
            if isinstance(data, dict) and data.get("object") == "list":
                return "ok", ""
        except Exception:
            pass
        return "non_json", "HTTP 200 nhưng phản hồi không phải JSON danh sách model"
    return "http_error", f"HTTP {r.status_code}"


def launch_cloudflared(
    extra_args,
    ready_timeout: int = 40,
    want_url: bool = False,
    kill_existing: bool = True,
    log_path: str = CF_LOG,
):
    """Chạy cloudflared ngầm. Trả về (đã_kết_nối, quick_url_hoặc_None). Ném lỗi nếu tiến trình chết sớm."""
    if kill_existing:
        subprocess.run(["pkill", "-f", "cloudflared"], check=False)
        try:
            os.remove(CF_QUICK_LOG)
        except OSError:
            pass
        time.sleep(1)
    log_file = open(log_path, "w", encoding="utf-8")
    proc = subprocess.Popen(
        [CF_BIN, "tunnel", "--no-autoupdate", *extra_args],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print("⏳ Đang kết nối tới mạng biên Cloudflare...")
    t0 = time.time()
    registered = False
    quick_url = None
    while time.time() - t0 < ready_timeout:
        content = read_cf_log(log_path)
        if proc.poll() is not None:
            tail = "\n".join(content.splitlines()[-15:])
            raise RuntimeError(f"cloudflared dừng đột ngột (mã {proc.returncode}). Log cuối:\n{redact(tail)}")
        registered = registered or ("Registered tunnel connection" in content)
        if want_url and not quick_url:
            m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", content)
            quick_url = m.group(0) if m else None
        if registered and (not want_url or quick_url):
            break
        time.sleep(1.0)
    return registered, quick_url


def start_cloudflare_tunnel(port: int) -> str:
    """Tên miền cố định nếu token dùng được; nếu bị Cloudflare Access chặn hoặc lỗi thì (tuỳ chọn) mở thêm link tạm trycloudflare.com."""
    ensure_cloudflared()
    host = CLOUDFLARE_HOSTNAME.strip().rstrip("/")
    if not host.startswith("http"):
        host = f"https://{host}"

    candidates = cloudflare_token_candidates()
    if not candidates:
        print("⚠️ Không có CLOUDFLARE_TOKEN hợp lệ (ô nhập tay hoặc 🔑 Secrets).")

    named_tunnel_up = False
    for label, kind, value in candidates:
        try:
            kind_name = "Tunnel connector token" if kind == "tunnel" else "API token"
            print(f"🔑 Đang dùng {kind_name} từ {label}...")
            connector_token = resolve_cloudflare_connector_token(kind, value, host, port)
            registered, _ = launch_cloudflared(["run", "--token", connector_token])
            if registered:
                print(f"✅ Cloudflare Tunnel đã kết nối (Healthy) -> {host}")
            else:
                print(f"⚠️ cloudflared đang chạy ngầm -> {host} nhưng chưa thấy 'Registered' sau 40s (xem trạng thái trên Cloudflare Dashboard).")
            named_tunnel_up = True
            break
        except Exception as exc:
            print(f"❌ Không dùng được token từ {label}: {redact(exc)}")

    if named_tunnel_up:
        cf_access_blocked = None
        for _ in range(4):
            try:
                status, detail = probe_public_models_api(host, timeout_s=6.0)
                if status == "ok":
                    return host
                if status == "cloudflare_access":
                    cf_access_blocked = detail
                    break
            except Exception:
                pass
            time.sleep(2.0)

        if cf_access_blocked is None:
            return host

        hostname_only = urlparse(host).hostname or host
        print("=" * 78)
        print(f"❌ PHÁT HIỆN CLOUDFLARE ZERO TRUST ACCESS ĐANG CHẶN '{host}'!")
        print(f"   Mọi API call đều bị HTTP 302 chuyển hướng sang trang login HTML: {cf_access_blocked}")
        print("   (Đây là nguyên nhân khiến ainovel-cli / OpenAI SDK báo lỗi 'openai: decode response'!)")
        print(f"   🔧 CÁCH SỬA ĐỂ DÙNG '{host}' (chỉ mất 30 giây, KHÔNG cần chạy lại Colab):")
        print("      1. Vào Cloudflare Zero Trust Dashboard -> Access -> Applications")
        print(f"         (hoặc Networks -> Tunnels -> {CF_TUNNEL_NAME} -> Public Hostname -> {hostname_only}).")
        print(f"      2. Tắt 'Protect with Access' / xóa Access Policy đang áp lên '{hostname_only}'.")
        print("         (FastAPI Gateway đã tự kiểm tra Bearer API Key nên không cần lớp login HTML này.)")
        print("=" * 78)
        if not FALLBACK_TO_QUICK_TUNNEL:
            raise RuntimeError(f"{host} đang bị Cloudflare Zero Trust Access chặn (HTTP 302 -> {cf_access_blocked}).")

        print(f"🔄 Mở thêm Cloudflare Quick Tunnel dự phòng (vẫn giữ song song tunnel '{host}' chạy ngầm)...")
        _, url = launch_cloudflared(
            ["--url", f"http://127.0.0.1:{port}"],
            ready_timeout=60,
            want_url=True,
            kill_existing=False,
            log_path=CF_QUICK_LOG,
        )
        if not url:
            raise RuntimeError("Không lấy được link trycloudflare.com. Log cuối:\n" + redact("\n".join(read_cf_log(CF_QUICK_LOG).splitlines()[-15:])))
        print(f"✅ Link Quick Tunnel dùng ngay không bị chặn: {url}")
        print(f"💡 Tên miền cố định {host} vẫn đang chạy song song — ngay khi bạn tắt Cloudflare Access trên Dashboard là dùng lại {host}/v1 được ngay!")
        return url.rstrip("/")

    if not FALLBACK_TO_QUICK_TUNNEL:
        raise RuntimeError("Không dựng được tunnel tên miền cố định và FALLBACK_TO_QUICK_TUNNEL=False.")

    print("🔄 Chuyển sang Cloudflare Quick Tunnel (không cần token, URL đổi mỗi lần chạy)...")
    registered, url = launch_cloudflared(["--url", f"http://127.0.0.1:{port}"], ready_timeout=60, want_url=True)
    if not url:
        raise RuntimeError("Không lấy được link trycloudflare.com. Log cuối:\n" + redact("\n".join(read_cf_log().splitlines()[-15:])))
    print(f"⚠️ Đang dùng link TẠM: {url}")
    print(f"   {host} sẽ CHƯA hoạt động cho tới khi bạn sửa token. Quick Tunnel có thể không hỗ trợ tốt streaming (SSE).")
    return url.rstrip("/")


def start_gradio_live_tunnel(port: int) -> str:
    """Tạo đường hầm *.gradio.live trỏ thẳng vào FastAPI Gateway."""
    try:
        import gradio  # noqa: F401
    except ImportError:
        print("📦 Đang cài đặt gradio để tạo tunnel gradio.live...")
        subprocess.run([sys.executable, "-m", "pip", "install", "gradio", "--quiet"], check=True)

    from gradio.networking import setup_tunnel

    subprocess.run(["pkill", "-f", "frpc_"], check=False)
    time.sleep(0.5)

    sig = inspect.signature(setup_tunnel)
    kwargs = {
        "local_host": "127.0.0.1",
        "local_port": int(port),
        "share_token": secrets.token_urlsafe(32),
        "share_server_address": None,
    }
    if "share_server_tls_certificate" in sig.parameters:
        kwargs["share_server_tls_certificate"] = None

    url = setup_tunnel(**kwargs)
    return str(url).rstrip("/")


def start_ngrok_tunnel(port: int) -> str:
    """Tạo đường hầm Ngrok (cần NGROK_TOKEN trong Colab Secrets)."""
    from pyngrok import ngrok

    ngrok_token = read_secret("NGROK_TOKEN")
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
    else:
        print("⚠️ Không tìm thấy NGROK_TOKEN trong Secrets của Colab!")

    ngrok.kill()
    tunnel = ngrok.connect(int(port))
    return str(tunnel.public_url).rstrip("/")


def publish_discovery_endpoint(base_v1_url: str, provider: str):
    """Ghi URL hiện tại lên Hugging Face của CHÍNH BẠN. Bước phụ: lỗi chỉ cảnh báo, không bao giờ làm dừng cell."""
    try:
        payload = {
            "base_url": base_v1_url,
            "public_url": base_v1_url[:-3] if base_v1_url.endswith("/v1") else base_v1_url,
            "tunnel_provider": provider,
            "updated_at": int(time.time()),
        }
        local_json = "/content/claw_endpoint.json"
        with open(local_json, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)

        hf_token = read_secret("HF_TOKEN").strip()
        if not hf_token:
            print("ℹ️ Bỏ qua đồng bộ Hugging Face (chưa có HF_TOKEN trong 🔑 Secrets).")
            return None

        api = HfApi(token=hf_token)
        me = api.whoami().get("name")
        repo = (DISCOVERY_HF_REPO or "").strip()
        if not repo or repo.split("/")[0] != me:
            if repo:
                print(f"ℹ️ Repo '{repo}' không thuộc tài khoản '{me}' -> tự dùng '{me}/claw-endpoint'.")
            repo = f"{me}/claw-endpoint"

        api.create_repo(repo, repo_type="model", private=False, exist_ok=True)
        api.upload_file(
            path_or_fileobj=local_json,
            path_in_repo=DISCOVERY_FILE_IN_REPO,
            repo_id=repo,
            repo_type="model",
            commit_message=f"Auto-update Colab endpoint ({provider})",
        )
        raw_url = f"https://huggingface.co/{repo}/resolve/main/{DISCOVERY_FILE_IN_REPO}"
        print(f"✅ Đã đồng bộ endpoint lên Hugging Face: {raw_url}")
        return raw_url
    except Exception as exc:
        print(f"⚠️ Bỏ qua đồng bộ Hugging Face: {exc}\n"
              "   -> Cần HF_TOKEN loại *Write* (tạo tại huggingface.co/settings/tokens). Không ảnh hưởng tới API đang chạy.")
        return None


def gateway_is_up(port) -> bool:
    try:
        r = httpx.get(f"http://127.0.0.1:{port}/health", headers={"User-Agent": "colab-selftest/1.0"}, timeout=3)
        return r.status_code == 200
    except Exception:
        return False


def require_local_gateway(port) -> None:
    """Không mở tunnel khi gateway chưa chạy trên runtime NÀY.

    Nếu cùng một tunnel có 2 runtime kết nối, Cloudflare chia lưu lượng cho cả hai và các request rơi vào
    runtime không có gateway sẽ lỗi, trong khi kiểm tra từ bên ngoài vẫn có thể báo 'OK' nhờ runtime kia.
    """
    for _ in range(5):
        if gateway_is_up(port):
            print(f"✅ Gateway nội bộ trên runtime này đang chạy (port {port}).")
            return
        time.sleep(2)
    raise RuntimeError(
        f"Gateway chưa chạy trên runtime này (127.0.0.1:{port} từ chối kết nối) nên cell 4 dừng, chưa mở tunnel.\n"
        "  - Runtime mới/vừa khởi động lại: chạy lần lượt cell 1 -> 2 -> 3 rồi mới chạy cell 4.\n"
        "  - Nếu service đang chạy ở runtime/tab Colab khác: KHÔNG chạy cell 4 ở đây, vì 2 runtime cùng nối vào một tunnel "
        "thì một phần request sẽ rơi vào runtime này và lỗi."
    )


def self_test(public_url: str, port) -> None:
    """Kiểm tra nhanh gateway nội bộ và xác thực URL công khai trả về đúng JSON OpenAI API."""
    local_ok = gateway_is_up(port)
    print(f"{'✅' if local_ok else '❌'} Gateway nội bộ trên runtime này: {'đang chạy' if local_ok else 'KHÔNG chạy'}")

    url = f"{public_url.rstrip('/')}/v1/models"
    last = "chưa thử"
    for _ in range(8):
        try:
            status, detail = probe_public_models_api(public_url, timeout_s=8.0)
            if status == "ok":
                print(f"✅ URL công khai hoạt động (JSON OpenAI API hợp lệ): {url}")
                if not local_ok:
                    print("⚠️ Nhưng gateway của runtime này không chạy, nên câu trả lời đến từ một runtime KHÁC đang nối vào cùng tunnel. "
                          "Hãy tắt runtime thừa (Runtime -> Manage sessions) để tránh lỗi chập chờn.")
                return
            if status == "cloudflare_access":
                print(f"❌ URL công khai '{url}' đang bị Cloudflare Zero Trust Access chặn (HTTP 302 -> {detail})!")
                return
            last = detail or status
        except Exception as exc:
            last = type(exc).__name__
        time.sleep(4)
    print(f"⚠️ Chưa gọi được JSON từ {url} ({last}). DNS/Cloudflare có thể cần thêm 1-2 phút; hãy thử lại bằng curl từ máy của bạn.")


# ============================== CHẠY CHÍNH ==============================
if __name__ == "__main__" or "get_ipython" in globals():
    require_local_gateway(GATEWAY_PORT)
    if TUNNEL_PROVIDER == "cloudflare":
        print(f"🌐 Đang khởi tạo Cloudflare Tunnel cho {CLOUDFLARE_HOSTNAME}...")
        public_url = start_cloudflare_tunnel(int(GATEWAY_PORT))
    elif TUNNEL_PROVIDER == "gradio":
        print("🌐 Đang khởi tạo đường hầm công khai qua gradio.live...")
        public_url = start_gradio_live_tunnel(int(GATEWAY_PORT))
    else:
        print("🌐 Đang khởi tạo đường hầm công khai qua Ngrok...")
        public_url = start_ngrok_tunnel(int(GATEWAY_PORT))

    base_v1_url = f"{public_url}/v1"
    publish_discovery_endpoint(base_v1_url, TUNNEL_PROVIDER)
    self_test(public_url, GATEWAY_PORT)

    print("=" * 78)
    print(f"👉 BASE_URL CHO CẢ LLM & STT ({TUNNEL_PROVIDER.upper()}): {base_v1_url}")
    print("=" * 78)
    print("1. Kiểm tra danh sách model (PowerShell):")
    print(f'   curl.exe -i "{base_v1_url}/models"')
    print("\n2. Chat với LLM 26B:")
    print(f"   POST {base_v1_url}/chat/completions (model='main-llm' hoặc '{LLM_MODEL_FILE}')")
    print("\n3. Chuyển giọng nói thành văn bản (Gemma 4 E2B STT):")
    print(f"   POST {base_v1_url}/audio/transcriptions (model='gemma-4-stt', file=voice.wav)")
    print("=" * 78)

    # Stream đồng thời log của Gateway (8000), LLM (8001), STT (8003) và cloudflared
    log_files = []
    for tag, path in (
        ("GATEWAY:8000", "/content/gateway_8000.log"),
        ("LLM-26B:8001", "/content/llm_8001.log"),
        ("STT-E2B:8003", "/content/stt_8003.log"),
        ("CLOUDFLARED", CF_LOG),
    ):
        if os.path.exists(path):
            lf = open(path, "r", encoding="utf-8", errors="replace")
            lf.seek(0, 2)  # chỉ đọc phần log mới
            log_files.append((tag, lf))
        else:
            print(f"ℹ️ Không thấy {path}, bỏ qua log {tag}.")
    if os.path.exists(CF_QUICK_LOG):
        lf_quick = open(CF_QUICK_LOG, "r", encoding="utf-8", errors="replace")
        lf_quick.seek(0, 2)
        log_files.append(("CLOUDFLARED-QUICK", lf_quick))

    print("\n📡 Đang theo dõi log trực tiếp (bấm Stop để thoát; các service vẫn chạy ngầm)...\n")
    try:
        while True:
            had_output = False
            for tag, lf in log_files:
                line = lf.readline()
                if line:
                    sys.stdout.write(f"[{tag}] {redact(line)}")
                    sys.stdout.flush()
                    had_output = True
            if not had_output:
                time.sleep(0.2)
    except KeyboardInterrupt:
        print("\nĐã dừng theo dõi log (các service vẫn đang chạy ngầm).")

✅ Gateway nội bộ trên runtime này đang chạy (port 8000).
🌐 Đang khởi tạo Cloudflare Tunnel cho https://api.gloryvocab.com...
🔑 Đang dùng Tunnel connector token từ ô CLOUDFLARE_TOKEN_INPUT...
⏳ Đang kết nối tới mạng biên Cloudflare...
✅ Cloudflare Tunnel đã kết nối (Healthy) -> https://api.gloryvocab.com
✅ Đã đồng bộ endpoint lên Hugging Face: https://huggingface.co/hinody/claw-endpoint/resolve/main/claw_endpoint.json
✅ Gateway nội bộ trên runtime này: đang chạy
✅ URL công khai hoạt động: https://api.gloryvocab.com/v1/models
👉 BASE_URL CHO CẢ LLM & STT (CLOUDFLARE): https://api.gloryvocab.com/v1
1. Kiểm tra danh sách model (PowerShell):
   curl.exe -i "https://api.gloryvocab.com/v1/models"

2. Chat với LLM 26B:
   POST https://api.gloryvocab.com/v1/chat/completions (model='main-llm' hoặc 'gemma-4-26B-A4B-it-Q4_K_M.gguf')

3. Chuyển giọng nói thành văn bản (Gemma 4 E2B STT):
   POST https://api.gloryvocab.com/v1/audio/transcriptions (model='gemma-4-stt', file=voice.wav)

📡 Đang theo dõi 